# Sentiment Model Comparison: DistilBERT vs RoBERTa vs Twitter-RoBERTa

## Purpose

This notebook provides a **systematic comparative analysis of transformer-based sentiment classification models**, focusing on both:

- **Model architecture differences (general vs domain-specific)**
- **Data scaling behavior across multiple dataset sizes**

The models evaluated include:
- DistilBERT (lightweight general-purpose transformer)
- RoBERTa-base (larger general-purpose transformer)
- Twitter-RoBERTa (domain-specific transformer trained on Twitter data)

The primary objective is to understand how **model capacity and domain alignment jointly influence performance, efficiency, and scalability** in sentiment analysis tasks.

---

## What This Notebook Does

- Loads pre-computed experimental results from all models:
  - DistilBERT experiments
  - RoBERTa-base experiments
  - Twitter-RoBERTa experiments

- Performs comparative analysis across:
  - Small dataset (~100k samples)
  - Medium dataset (~200k samples)
  - Large dataset (~400k samples)

- Evaluates and compares models using:
  - Accuracy  
  - F1-score  
  - ROC-AUC  
  - Training time  

- Generates research-grade visualizations:
  - Performance vs dataset size
  - Model efficiency trade-offs
  - Scaling behavior curves
  - Performance gap analysis

---

## Key Research Focus

This notebook is designed to answer the following core questions:

- How does **model size affect sentiment classification performance**?
- Does **domain-specific pretraining (Twitter-RoBERTa)** provide consistent advantages over general models?
- How do models behave under **dataset scaling conditions**?
- What is the trade-off between:
  - Performance improvement  
  - Computational cost  
  - Data efficiency  

---

## Role in Overall Project

This notebook represents the **final integration stage of Phase 1 (comparative study)** and serves as a bridge toward Phase 2 (cross-domain analysis).

It consolidates results from previous experiments and is used to:

- Compare **DistilBERT vs RoBERTa vs Twitter-RoBERTa**
- Analyze **scaling behavior across architectures**
- Evaluate **data efficiency vs model complexity**
- Identify optimal trade-offs for real-world sentiment systems

---

## Key Research Contributions

This analysis aims to provide insights into:

> How transformer model performance is jointly influenced by architectural capacity, domain-specific pretraining, and dataset scale in sentiment classification tasks.

Specifically, it highlights:

- The role of **domain alignment in improving data efficiency**
- The impact of **model size on performance ceilings**
- The presence of **diminishing returns in dataset scaling**
- The trade-off between **accuracy and computational cost**

---

## Connection to Overall Study

This notebook directly connects to:

- Baseline ML models (Logistic Regression + TF-IDF)
- DistilBERT experiments (general transformer baseline)
- Twitter-RoBERTa experiments (domain-specific model study)

It forms the **central comparative layer** of the entire sentiment analysis research pipeline and supports the transition toward:

> Cross-domain sentiment analysis and brand-level sentiment modeling (Phase 2)

In [1]:
!uv pip install transformers datasets tqdm accelerate

Using Python 3.12.12 environment at: /usr
Audited 4 packages in 302ms


In [2]:
# !pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -U transformers accelerate

Looking in indexes: https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 91.6 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transf

Checking GPU

In [3]:
import torch
torch.cuda.is_available()

True

In [4]:
# !nvidia-smi

In [5]:
# ! pip install transformers[torch] datasets tqdm accelerate --only-binary :all: -i https://pypi.tuna.tsinghua.edu.cn/simple

# Dependencies

In [6]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, pipeline, BertTokenizerFast
from transformers import (
    DistilBertForSequenceClassification, 
    DistilBertTokenizerFast,
    AutoTokenizer,
    AutoModelForSequenceClassification
)
import time
import os
import json, pickle as pkl



# Loading Dataset

In [7]:
# ! pip install kaggle

get kaggle.json from the kaggle and add to working environment

In [8]:
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

Importing twitter sentiment dataset

In [9]:
# !kaggle datasets download -d kazanova/sentiment140

if data set is downloaded using: !kaggle datasets download -d kazanova/sentiment140

In [10]:
# # extracting the compressed dataset

# from zipfile import ZipFile
# dataset = '/content/sentiment140.zip'

# with ZipFile(dataset, 'r') as zip:
#   zip.extractall()
#   print('The dataset is extracted successfully')

Use datasets/kazanova/sentiment140 from kaggle

In [11]:
# colab
# df = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1', header = None)

#colab + dataset saved on drive
# df = pd.read_csv('/content/drive/MyDrive/Sentiment Analysis Project/datasets/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1', header = None)

#kaggle
df = pd.read_csv(
    '/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv',
    encoding='ISO-8859-1',
    header=None
)

df.columns = ['target', 'id', 'date', 'flag', 'user', 'text']

# Converting labels
df['target'] = df['target'].replace(4,1)

# Keeping only needed columns
df = df[['text', 'target']]


## Create 3 datasets

- Dataset 1: Small (50k per class → 100k total)
- Dataset 2: Medium (100k per class → 200k total) MAIN
- Dataset 3: Large (200k per class → 400k total)

In [12]:
df_small = df.groupby('target').sample(50000, random_state=42).reset_index(drop=True)

In [13]:
df_medium = df.groupby('target').sample(100000, random_state=42).reset_index(drop=True)

In [14]:
df_large = df.groupby('target').sample(200000, random_state=42).reset_index(drop=True)

Sanity check

In [15]:
print("Small:\n", df_small['target'].value_counts())
print("\nMedium:\n", df_medium['target'].value_counts())
print("\nLarge:\n", df_large['target'].value_counts())

Small:
 target
0    50000
1    50000
Name: count, dtype: int64

Medium:
 target
0    100000
1    100000
Name: count, dtype: int64

Large:
 target
0    200000
1    200000
Name: count, dtype: int64


In [16]:
# Uncomment below code to save the datasets

# df_small.to_csv("sentiment_small.csv", index=False)
# df_medium.to_csv("sentiment_medium.csv", index=False)
# df_large.to_csv("sentiment_large.csv", index=False)

# df_small.to_csv("/content/drive/MyDrive/Sentiment Analysis Project/datasets/sentiment_small.csv", index=False)
# df_medium.to_csv("/content/drive/MyDrive/Sentiment Analysis Project/datasets/sentiment_medium.csv", index=False)
# df_large.to_csv("/content/drive/MyDrive/Sentiment Analysis Project/datasets/sentiment_large.csv", index=False)

# REUSABLE Twitter-RoBERTa EXPERIMENT PIPELINE

## Metrics function

In [17]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=1)

  precision, recall,f1, _ = precision_recall_fscore_support(labels, preds, average = 'binary')
  acc = accuracy_score(labels, preds)

  probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
  roc = roc_auc_score(labels, probs)

  return {
      "accuracy": acc,
      "f1": f1,
      "roc_auc": roc,
      "precision": precision,
      "recall": recall
  }




MAIN reusable function

kaggle

Code suitable to plot:
- loss curves
- error analysis (predictions saved)
- proper logging for research plots
- reproducible experiment setup

In [18]:
def run_experiment(
    df,
    dataset_name="dataset",
    save_dir="/kaggle/working/models",
    model_name="cardiffnlp/twitter-roberta-base-sentiment"
):

    print(f"Running experiment on {dataset_name} using {model_name}")

    # Save directory
    exp_path = os.path.join(save_dir, dataset_name)
    os.makedirs(exp_path, exist_ok=True)

    # Train / Validation Split
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['text'].tolist(),
        df['target'].tolist(),
        test_size=0.2,
        random_state=42,
        stratify=df['target']
    )

    with open(os.path.join(exp_path, "data_split_info.json"), "w") as f:
        json.dump({
            "train_size": len(train_texts),
            "val_size": len(val_texts)
        }, f, indent=4)

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_encodings = tokenizer(
        train_texts,
        truncation=True,
        padding=True,
        max_length=128
    )

    val_encodings = tokenizer(
        val_texts,
        truncation=True,
        padding=True,
        max_length=128
    )

    # Dataset Class
    class SentimentDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels

        def __getitem__(self, idx):
            item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
            item["labels"] = torch.tensor(self.labels[idx])
            return item

        def __len__(self):
            return len(self.labels)

    train_dataset = SentimentDataset(train_encodings, train_labels)
    val_dataset = SentimentDataset(val_encodings, val_labels)

    # Model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        ignore_mismatched_sizes=True
    )

    # Training Arguments
    training_args = TrainingArguments(
        output_dir=os.path.join(exp_path, "checkpoints"),

        num_train_epochs=2,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,

        gradient_accumulation_steps=4,
        learning_rate=2e-5,

        eval_strategy="epoch",
        save_strategy="no",

        fp16=True,
        optim="adamw_torch",

        dataloader_num_workers=2,
        dataloader_pin_memory=True,

        logging_strategy="steps",
        logging_steps=100,

        report_to="none"
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    # Training
    start = time.time()
    trainer.train()
    end = time.time()

    trainer.save_state()

    # EVALUATION
    results = trainer.evaluate()

    preds = trainer.predict(val_dataset)

    logits = preds.predictions
    y_true = preds.label_ids
    y_pred = logits.argmax(axis=1)

    # PROBABILITIES (for analysis)
    import numpy as np
    from scipy.special import softmax

    probs = softmax(logits, axis=1)
    confidence = np.max(probs, axis=1)

    # PREDICTION DATASET (ERROR ANALYSIS READY)
    import pandas as pd

    df_preds = pd.DataFrame({
        "text": val_texts,
        "true": y_true,
        "pred": y_pred,
        "confidence": confidence
    })

    df_preds.to_csv(os.path.join(exp_path, "predictions.csv"), index=False)

    # LOSS LOGS (FOR CURVES)
    log_history = trainer.state.log_history

    steps = []
    train_loss = []
    eval_loss = []

    for log in log_history:
        if "loss" in log:
            steps.append(log["step"])
            train_loss.append(log["loss"])

        if "eval_loss" in log:
            eval_loss.append(log["eval_loss"])

    with open(os.path.join(exp_path, "loss_logs.json"), "w") as f:
        json.dump({
            "steps": steps,
            "train_loss": train_loss,
            "eval_loss": eval_loss
        }, f, indent=4)

    # CONFUSION MATRIX
    from sklearn.metrics import confusion_matrix, classification_report

    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, output_dict=True)

    with open(os.path.join(exp_path, "confusion_matrix.json"), "w") as f:
        json.dump(cm.tolist(), f, indent=4)

    with open(os.path.join(exp_path, "classification_report.json"), "w") as f:
        json.dump(report, f, indent=4)

    # FINAL RESULTS
    final_results = {
        "dataset": dataset_name,
        "model": model_name,
        "accuracy": results.get("eval_accuracy"),
        "f1": results.get("eval_f1"),
        "roc_auc": results.get("eval_roc_auc"),
        "training_time_sec": end - start
    }

    # SAVE EVERYTHING
    trainer.save_model(exp_path)
    tokenizer.save_pretrained(exp_path)

    with open(os.path.join(exp_path, "results.json"), "w") as f:
        json.dump(final_results, f, indent=4)

    import pickle as pkl
    with open(os.path.join(exp_path, "results.pkl"), "wb") as f:
        pkl.dump(final_results, f)

    with open(os.path.join(exp_path, "training_args.json"), "w") as f:
        json.dump(training_args.to_dict(), f, indent=4)

    label_map = {0: "negative", 1: "positive"}
    with open(os.path.join(exp_path, "label_map.json"), "w") as f:
        json.dump(label_map, f, indent=4)

    # SUMMARY
    print(f"\nExperiment saved at: {exp_path}")

    print("\n--- Files Generated ---")
    for root, dirs, files in os.walk(exp_path):
        for file in files:
            print(os.path.join(root, file))

    return final_results

# Run all experiments

In [19]:
import transformers
print(transformers.__version__)

5.7.0


In [20]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

CUDA available: True
GPU name: Tesla T4


In [21]:
# !nvidia-smi

In [22]:
TRAIN_SMALL = True
TRAIN_MEDIUM = True
TRAIN_LARGE = True



In [23]:

if TRAIN_LARGE:
    results_large = run_experiment(
        df_large,
        dataset_name="df_large-roberta",
        model_name="roberta-base"
    )
    print(results_large)

Running experiment on df_large-roberta using roberta-base


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1,Roc Auc,Precision,Recall
2.050795,0.556143,2,0.887312,0.886537,0.955662,0.892682,0.880475


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Experiment saved at: /kaggle/working/models/df_large-roberta

--- Files Generated ---
/kaggle/working/models/df_large-roberta/label_map.json
/kaggle/working/models/df_large-roberta/data_split_info.json
/kaggle/working/models/df_large-roberta/model.safetensors
/kaggle/working/models/df_large-roberta/results.json
/kaggle/working/models/df_large-roberta/results.pkl
/kaggle/working/models/df_large-roberta/config.json
/kaggle/working/models/df_large-roberta/tokenizer.json
/kaggle/working/models/df_large-roberta/classification_report.json
/kaggle/working/models/df_large-roberta/predictions.csv
/kaggle/working/models/df_large-roberta/tokenizer_config.json
/kaggle/working/models/df_large-roberta/training_args.json
/kaggle/working/models/df_large-roberta/training_args.bin
/kaggle/working/models/df_large-roberta/confusion_matrix.json
/kaggle/working/models/df_large-roberta/loss_logs.json
/kaggle/working/models/df_large-roberta/checkpoints/trainer_state.json
{'dataset': 'df_large-roberta', 'mode

In [24]:

if TRAIN_MEDIUM:
    results_medium = run_experiment(
        df_medium,
        dataset_name="df_medium-roberta",
        model_name="roberta-base"
    )
    print(results_medium)



Running experiment on df_medium-roberta using roberta-base


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vect

Epoch,Training Loss,Validation Loss,Accuracy,F1,Roc Auc,Precision,Recall
1,2.441068,0.586764,0.880750,0.881207,0.949892,0.877841,0.884600
2,2.191314,0.576599,0.883300,0.883189,0.952543,0.884030,0.882350


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1,Roc Auc,Precision,Recall
2.191314,0.576599,2,0.883300,0.883189,0.952543,0.884030,0.882350


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Experiment saved at: /kaggle/working/models/df_medium-roberta

--- Files Generated ---
/kaggle/working/models/df_medium-roberta/label_map.json
/kaggle/working/models/df_medium-roberta/data_split_info.json
/kaggle/working/models/df_medium-roberta/model.safetensors
/kaggle/working/models/df_medium-roberta/results.json
/kaggle/working/models/df_medium-roberta/results.pkl
/kaggle/working/models/df_medium-roberta/config.json
/kaggle/working/models/df_medium-roberta/tokenizer.json
/kaggle/working/models/df_medium-roberta/classification_report.json
/kaggle/working/models/df_medium-roberta/predictions.csv
/kaggle/working/models/df_medium-roberta/tokenizer_config.json
/kaggle/working/models/df_medium-roberta/training_args.json
/kaggle/working/models/df_medium-roberta/training_args.bin
/kaggle/working/models/df_medium-roberta/confusion_matrix.json
/kaggle/working/models/df_medium-roberta/loss_logs.json
/kaggle/working/models/df_medium-roberta/checkpoints/trainer_state.json
{'dataset': 'df_mediu

In [25]:
if TRAIN_SMALL:
    results_small = run_experiment(
        df_small,
        dataset_name="df_small-roberta",
        model_name="roberta-base"
    )
    print(results_small)



Running experiment on df_small-roberta using roberta-base


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vect

Epoch,Training Loss,Validation Loss,Accuracy,F1,Roc Auc,Precision,Recall
1,2.653232,0.619013,0.868150,0.866758,0.941986,0.876009,0.857700
2,2.297003,0.616467,0.874500,0.874374,0.946046,0.875251,0.873500


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,F1,Roc Auc,Precision,Recall
2.297003,0.616467,2,0.874500,0.874374,0.946046,0.875251,0.873500


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Experiment saved at: /kaggle/working/models/df_small-roberta

--- Files Generated ---
/kaggle/working/models/df_small-roberta/label_map.json
/kaggle/working/models/df_small-roberta/data_split_info.json
/kaggle/working/models/df_small-roberta/model.safetensors
/kaggle/working/models/df_small-roberta/results.json
/kaggle/working/models/df_small-roberta/results.pkl
/kaggle/working/models/df_small-roberta/config.json
/kaggle/working/models/df_small-roberta/tokenizer.json
/kaggle/working/models/df_small-roberta/classification_report.json
/kaggle/working/models/df_small-roberta/predictions.csv
/kaggle/working/models/df_small-roberta/tokenizer_config.json
/kaggle/working/models/df_small-roberta/training_args.json
/kaggle/working/models/df_small-roberta/training_args.bin
/kaggle/working/models/df_small-roberta/confusion_matrix.json
/kaggle/working/models/df_small-roberta/loss_logs.json
/kaggle/working/models/df_small-roberta/checkpoints/trainer_state.json
{'dataset': 'df_small-roberta', 'mode

# Compare results

In [26]:
# results_df = pd.DataFrame([
#     results_small,
#     results_medium,
#     results_large
# ])

# results_df

# Plots